<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment2/audio_assignment_extract_birdnet_embeddings_speedup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pip install -q "birdnet[and-cuda]" scikit-learn pandas numpy tqdm joblib soundfile

In [13]:
from pathlib import Path
import shutil
import os

PROJECT_DIR = Path("/content/drive/MyDrive/audio_assignment/zip")

DRIVE_AUDIO_ZIP = PROJECT_DIR / "train_audio.zip"
LOCAL_ROOT = Path("/content/audio_assignment_runtime")
LOCAL_ZIP = Path("/content/train_audio.zip")

OUTPUT_DIR = PROJECT_DIR / "outputs_a1_birdnet_ml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_DIR = PROJECT_DIR / "embeddings" / "birdnet_original_mean"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_AUDIO_ZIP.exists():
    raise FileNotFoundError(
        f"Expected zip file not found: {DRIVE_AUDIO_ZIP}\n"
        "Create train_audio.zip in Google Drive first."
    )

if not LOCAL_ROOT.exists():
    print("Copying train_audio.zip from Drive to local Colab disk...")
    shutil.copy2(DRIVE_AUDIO_ZIP, LOCAL_ZIP)

    print("Unzipping locally...")
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -q /content/train_audio.zip -d /content/audio_assignment_runtime
else:
    print("Local extracted folder already exists.")

# Find train_audio folder after unzip
matches = list(LOCAL_ROOT.rglob("train_audio"))

if len(matches) > 0:
    AUDIO_DIR = matches[0]
else:
    # If zip directly contains class folders
    AUDIO_DIR = LOCAL_ROOT

print("Project folder:", PROJECT_DIR)
print("Local audio folder:", AUDIO_DIR)
print("Embeddings folder:", EMBEDDINGS_DIR)
print("Audio folder exists:", AUDIO_DIR.exists())


Local extracted folder already exists.
Project folder: /content/drive/MyDrive/audio_assignment/zip
Local audio folder: /content/audio_assignment_runtime/train_audio
Embeddings folder: /content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean
Audio folder exists: True


In [14]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Keep this exactly the same as before if you want the same file count.
# Only add ".opus" if the ignored extension check shows real audio files with .opus.
AUDIO_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".aiff", ".aif"]
# AUDIO_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".aiff", ".aif", ".opus"]

allowed_exts = set([ext.lower() for ext in AUDIO_EXTENSIONS])

all_files_under_train_audio = [p for p in AUDIO_DIR.rglob("*") if p.is_file()]

accepted_files = []
ignored_files = []

for p in all_files_under_train_audio:
    ext = p.suffix.lower() if p.suffix else "[no extension]"

    if ext in allowed_exts:
        accepted_files.append(p)
    else:
        ignored_files.append(p)

print("Total files under train_audio:", len(all_files_under_train_audio))
print("Accepted audio files:", len(accepted_files))
print("Ignored files not in extension list:", len(ignored_files))

print("\nAllowed audio extensions:")
print(sorted(allowed_exts))

ignored_ext_counts = Counter(
    [p.suffix.lower() if p.suffix else "[no extension]" for p in ignored_files]
)

print("\nIgnored file extension counts:")
for ext, count in ignored_ext_counts.most_common():
    print(f"{ext}: {count}")

print("\nExample ignored files:")
for p in ignored_files[:30]:
    print(p)

ignored_df = pd.DataFrame({
    "filepath": [str(p) for p in ignored_files],
    "extension": [p.suffix.lower() if p.suffix else "[no extension]" for p in ignored_files]
})

ignored_csv_path = OUTPUT_DIR / "ignored_files_not_in_audio_extension_list.csv"
ignored_df.to_csv(ignored_csv_path, index=False)

print("\nIgnored file list saved to:")
print(ignored_csv_path)


# ----------------------------------------------------
# Create df for later train/validation split
# ----------------------------------------------------

rows = []
skipped_no_class_folder = []

for audio_path in accepted_files:
    relative_path = audio_path.relative_to(AUDIO_DIR)

    # Expected:
    # train_audio/class_name/audio_file.wav
    if len(relative_path.parts) < 2:
        print("WARNING: File has no class folder, skipping:")
        print(audio_path)

        skipped_no_class_folder.append({
            "filepath": str(audio_path),
            "relative_filepath": relative_path.as_posix(),
            "extension": audio_path.suffix.lower() if audio_path.suffix else "[no extension]"
        })
        continue

    label = relative_path.parts[0]

    rows.append({
        # Keep filepath for compatibility with your previous notebook.
        # In Colab this will be the local /content path after unzip.
        "filepath": str(audio_path),

        # Explicit local path used by BirdNET.
        "local_filepath": str(audio_path),

        # Portable path: class/file. This is useful when moving between Drive/local/Colab.
        "relative_filepath": relative_path.as_posix(),

        "label": str(label),
        "extension": audio_path.suffix.lower() if audio_path.suffix else "[no extension]",
        "file_size_bytes": audio_path.stat().st_size
    })

df = pd.DataFrame(rows)

skipped_df = pd.DataFrame(skipped_no_class_folder)
skipped_csv_path = OUTPUT_DIR / "skipped_files_no_class_folder.csv"
skipped_df.to_csv(skipped_csv_path, index=False)

print("\ndf created")
print("Total detected audio files in df:", len(df))
print("Total detected classes in df:", df["label"].nunique())
print("Skipped files with no class folder:", len(skipped_df))

display(df.head())

df_csv_path = OUTPUT_DIR / "all_detected_audio_files.csv"
df.to_csv(df_csv_path, index=False)

print("\nDetected audio file list saved to:")
print(df_csv_path)

print("\nSkipped no-class-folder file list saved to:")
print(skipped_csv_path)

print("\nTop 20 class counts:")
display(df["label"].value_counts().head(20))

Total files under train_audio: 35549
Accepted audio files: 35549
Ignored files not in extension list: 0

Allowed audio extensions:
['.aif', '.aiff', '.flac', '.mp3', '.ogg', '.wav']

Ignored file extension counts:

Example ignored files:

Ignored file list saved to:
/content/drive/MyDrive/audio_assignment/zip/outputs_a1_birdnet_ml/ignored_files_not_in_audio_extension_list.csv

df created
Total detected audio files in df: 35549
Total detected classes in df: 206
Skipped files with no class folder: 0


,filepath,local_filepath,relative_filepath,label,extension,file_size_bytes
0,/content/audio_assignment_runtime/train_audio/...,/content/audio_assignment_runtime/train_audio/...,fepowl/iNat1664968.ogg,fepowl,.ogg,5389
1,/content/audio_assignment_runtime/train_audio/...,/content/audio_assignment_runtime/train_audio/...,fepowl/iNat133178.ogg,fepowl,.ogg,169268
2,/content/audio_assignment_runtime/train_audio/...,/content/audio_assignment_runtime/train_audio/...,fepowl/XC427412.ogg,fepowl,.ogg,54994
3,/content/audio_assignment_runtime/train_audio/...,/content/audio_assignment_runtime/train_audio/...,fepowl/XC727674.ogg,fepowl,.ogg,98212
4,/content/audio_assignment_runtime/train_audio/...,/content/audio_assignment_runtime/train_audio/...,fepowl/XC325660.ogg,fepowl,.ogg,629801



Detected audio file list saved to:
/content/drive/MyDrive/audio_assignment/zip/outputs_a1_birdnet_ml/all_detected_audio_files.csv

Skipped no-class-folder file list saved to:
/content/drive/MyDrive/audio_assignment/zip/outputs_a1_birdnet_ml/skipped_files_no_class_folder.csv

Top 20 class counts:


,count
label,
rubthr1,499
banana,498
fepowl,497
soulap1,497
houspa,496
coffal1,495
osprey,495
socfly1,494
compau,493


In [15]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import numpy as np

RANDOM_STATE = 42

# Store split outside individual experiment folder
# so A1, A2, B1, B2 all use the same validation set.
SPLIT_DIR = PROJECT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = SPLIT_DIR / "train_validation_split.csv"

# Keep this False normally.
# Set to True only if you intentionally want to delete/recreate the split.
RESET_SPLIT = False


def make_relative_filepath_from_path(path_value):
    """
    Converts an absolute path into a stable relative path.

    Example:
    /content/audio_assignment_runtime/train_audio/113949/file.ogg
    becomes:
    113949/file.ogg

    This allows the fixed split to work even when files move from
    Google Drive to local Colab /content storage.
    """

    path_str = str(path_value).replace("\\", "/").strip()

    if "/train_audio/" in path_str:
        relative_path = path_str.split("/train_audio/", 1)[1]
    elif path_str.startswith("train_audio/"):
        relative_path = path_str.split("train_audio/", 1)[1]
    else:
        # Fallback: assume it is already relative
        relative_path = path_str

    relative_path = relative_path.lstrip("/")

    return relative_path


def ensure_portable_file_columns(dataframe, audio_dir):
    """
    Ensures dataframe has:
    - relative_filepath: stable path used for split comparison
    - local_filepath: current Colab local path used by BirdNET
    """

    dataframe = dataframe.copy()

    if "filepath" not in dataframe.columns:
        raise ValueError("Expected dataframe to contain a 'filepath' column.")

    if "label" not in dataframe.columns:
        raise ValueError("Expected dataframe to contain a 'label' column.")

    dataframe["filepath"] = dataframe["filepath"].astype(str)
    dataframe["label"] = dataframe["label"].astype(str)

    if "relative_filepath" not in dataframe.columns:
        dataframe["relative_filepath"] = dataframe["filepath"].apply(
            make_relative_filepath_from_path
        )
    else:
        dataframe["relative_filepath"] = dataframe["relative_filepath"].fillna("").astype(str)

        blank_mask = dataframe["relative_filepath"].str.strip() == ""

        if blank_mask.any():
            dataframe.loc[blank_mask, "relative_filepath"] = dataframe.loc[
                blank_mask, "filepath"
            ].apply(make_relative_filepath_from_path)

    # Normalise relative filepath format
    dataframe["relative_filepath"] = (
        dataframe["relative_filepath"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .str.lstrip("/")
    )

    # If some values accidentally still start with train_audio/, remove that part
    dataframe["relative_filepath"] = dataframe["relative_filepath"].apply(
        lambda p: p.split("train_audio/", 1)[1] if p.startswith("train_audio/") else p
    )

    # This is the path BirdNET should use in Colab.
    # AUDIO_DIR should point to local unzipped train_audio folder under /content.
    dataframe["local_filepath"] = dataframe["relative_filepath"].apply(
        lambda rel: str(Path(audio_dir) / rel)
    )

    return dataframe


def create_fixed_train_validation_split(df, split_file):
    """
    Creates a fixed train/validation split.

    Rules:
    - Classes with only 1 sample are forced into validation.
    - Classes with 2 samples get 1 train and 1 validation.
    - Classes with more than 2 samples use approximately 80/20 split.
    - The resulting split is saved to CSV and reused in future runs.
    """

    df = ensure_portable_file_columns(df, AUDIO_DIR)

    counts = df["label"].value_counts()
    single_classes = counts[counts == 1].index.tolist()

    single_df = df[df["label"].isin(single_classes)].copy()
    multi_df = df[~df["label"].isin(single_classes)].copy()

    train_parts = []
    val_parts = []

    for label, group in multi_df.groupby("label"):
        # Sort by relative path so split is stable even if absolute path changes
        group = group.sort_values("relative_filepath").reset_index(drop=True)
        n = len(group)

        if n == 2:
            train_g, val_g = train_test_split(
                group,
                test_size=1,
                random_state=RANDOM_STATE,
                shuffle=True
            )
        else:
            train_g, val_g = train_test_split(
                group,
                test_size=0.2,
                random_state=RANDOM_STATE,
                shuffle=True
            )

        train_parts.append(train_g)
        val_parts.append(val_g)

    if len(train_parts) > 0:
        train_df_created = pd.concat(train_parts, ignore_index=True)
    else:
        train_df_created = pd.DataFrame(columns=df.columns)

    if len(val_parts) > 0:
        val_df_created = pd.concat(val_parts + [single_df], ignore_index=True)
    else:
        val_df_created = single_df.copy()

    train_df_created["split"] = "train"
    val_df_created["split"] = "validation"

    split_df_created = pd.concat(
        [train_df_created, val_df_created],
        ignore_index=True
    )

    split_df_created = split_df_created.sort_values(
        ["split", "label", "relative_filepath"]
    ).reset_index(drop=True)

    # Do not save local_filepath in the permanent split file.
    # local_filepath is runtime-specific and will be recreated each run.
    split_to_save = split_df_created.drop(
        columns=["local_filepath"],
        errors="ignore"
    )

    split_to_save.to_csv(split_file, index=False)

    return split_to_save


# ----------------------------------------------------
# Load existing split or create it once
# ----------------------------------------------------

# Make current scanned df portable first
df = ensure_portable_file_columns(df, AUDIO_DIR)

if SPLIT_FILE.exists() and not RESET_SPLIT:
    print("Existing split found. Reusing fixed split:")
    print(SPLIT_FILE)

    # Read all columns as strings so class labels like 116570 stay as strings
    split_df = pd.read_csv(SPLIT_FILE, dtype=str)

else:
    if RESET_SPLIT and SPLIT_FILE.exists():
        print("RESET_SPLIT=True, so recreating the split.")
    else:
        print("No existing split found. Creating split once.")

    split_df = create_fixed_train_validation_split(df, SPLIT_FILE)

# Make loaded/created split portable and add local_filepath for this Colab run
split_df = ensure_portable_file_columns(split_df, AUDIO_DIR)

print("Split loaded.")

display(split_df.head())
display(split_df["split"].value_counts())


# ----------------------------------------------------
# Create train and validation dataframes
# ----------------------------------------------------

train_df = split_df[split_df["split"] == "train"].copy().reset_index(drop=True)
val_df = split_df[split_df["split"] == "validation"].copy().reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Total samples:", len(split_df))

print("Train classes:", train_df["label"].nunique())
print("Validation classes:", val_df["label"].nunique())

single_example_classes = (
    split_df.groupby("label")
    .size()
    .loc[lambda x: x == 1]
    .index
    .tolist()
)

print("Single-example classes forced to validation:", len(single_example_classes))
print(single_example_classes[:20])


# ----------------------------------------------------
# Compare current dataset against saved split using relative paths
# ----------------------------------------------------

current_files = set(df["relative_filepath"].astype(str))
split_files = set(split_df["relative_filepath"].astype(str))

new_files_not_in_split = sorted(list(current_files - split_files))
missing_files_from_split = sorted(list(split_files - current_files))

print("New files not included in fixed split:", len(new_files_not_in_split))
print("Missing files from saved split:", len(missing_files_from_split))

if len(new_files_not_in_split) > 0:
    print("WARNING: These files are in the dataset folder but not in the saved split.")
    print("They will not be used unless you recreate the split intentionally.")
    print(new_files_not_in_split[:10])

if len(missing_files_from_split) > 0:
    print("WARNING: These files are in the saved split but missing from the dataset folder.")
    print("Check if files were moved, renamed, or deleted.")
    print(missing_files_from_split[:10])


# ----------------------------------------------------
# Check that local /content paths exist before BirdNET extraction
# ----------------------------------------------------

missing_train_local = train_df[
    ~train_df["local_filepath"].apply(lambda p: Path(p).exists())
].copy()

missing_val_local = val_df[
    ~val_df["local_filepath"].apply(lambda p: Path(p).exists())
].copy()

print("Missing local train files:", len(missing_train_local))
print("Missing local validation files:", len(missing_val_local))

if len(missing_train_local) > 0:
    print("Example missing train local files:")
    display(missing_train_local.head(10))

if len(missing_val_local) > 0:
    print("Example missing validation local files:")
    display(missing_val_local.head(10))

if len(missing_train_local) > 0 or len(missing_val_local) > 0:
    raise FileNotFoundError(
        "Some split files were not found in the local AUDIO_DIR. "
        "Check that train_audio.zip was unzipped correctly and AUDIO_DIR points to the train_audio folder."
    )


# ----------------------------------------------------
# Save runtime copies with local paths for debugging
# ----------------------------------------------------

runtime_train_path = OUTPUT_DIR / "runtime_train_split_with_local_paths.csv"
runtime_val_path = OUTPUT_DIR / "runtime_validation_split_with_local_paths.csv"

train_df.to_csv(runtime_train_path, index=False)
val_df.to_csv(runtime_val_path, index=False)

print("Runtime train split saved to:")
print(runtime_train_path)

print("Runtime validation split saved to:")
print(runtime_val_path)

print("\nExample train rows:")
display(train_df.head())

Existing split found. Reusing fixed split:
/content/drive/MyDrive/audio_assignment/zip/splits/train_validation_split.csv
Split loaded.


,filepath,relative_filepath,label,extension,file_size_bytes,split,local_filepath
0,/content/audio_assignment_runtime/train_audio/...,1161364/iNat1216197.ogg,1161364,.ogg,169832,train,/content/audio_assignment_runtime/train_audio/...
1,/content/audio_assignment_runtime/train_audio/...,1161364/iNat1264238.ogg,1161364,.ogg,7409,train,/content/audio_assignment_runtime/train_audio/...
2,/content/audio_assignment_runtime/train_audio/...,1161364/iNat547199.ogg,1161364,.ogg,159412,train,/content/audio_assignment_runtime/train_audio/...
3,/content/audio_assignment_runtime/train_audio/...,1161364/iNat556514.ogg,1161364,.ogg,692070,train,/content/audio_assignment_runtime/train_audio/...
4,/content/audio_assignment_runtime/train_audio/...,1161364/iNat818781.ogg,1161364,.ogg,576205,train,/content/audio_assignment_runtime/train_audio/...


,count
split,
train,28357
validation,7192


Train samples: 28357
Validation samples: 7192
Total samples: 35549
Train classes: 202
Validation classes: 206
Single-example classes forced to validation: 4
['116570', '23150', '23724', '516975']
New files not included in fixed split: 0
Missing files from saved split: 0
Missing local train files: 0
Missing local validation files: 0
Runtime train split saved to:
/content/drive/MyDrive/audio_assignment/zip/outputs_a1_birdnet_ml/runtime_train_split_with_local_paths.csv
Runtime validation split saved to:
/content/drive/MyDrive/audio_assignment/zip/outputs_a1_birdnet_ml/runtime_validation_split_with_local_paths.csv

Example train rows:


,filepath,relative_filepath,label,extension,file_size_bytes,split,local_filepath
0,/content/audio_assignment_runtime/train_audio/...,1161364/iNat1216197.ogg,1161364,.ogg,169832,train,/content/audio_assignment_runtime/train_audio/...
1,/content/audio_assignment_runtime/train_audio/...,1161364/iNat1264238.ogg,1161364,.ogg,7409,train,/content/audio_assignment_runtime/train_audio/...
2,/content/audio_assignment_runtime/train_audio/...,1161364/iNat547199.ogg,1161364,.ogg,159412,train,/content/audio_assignment_runtime/train_audio/...
3,/content/audio_assignment_runtime/train_audio/...,1161364/iNat556514.ogg,1161364,.ogg,692070,train,/content/audio_assignment_runtime/train_audio/...
4,/content/audio_assignment_runtime/train_audio/...,1161364/iNat818781.ogg,1161364,.ogg,576205,train,/content/audio_assignment_runtime/train_audio/...


In [16]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

!nvidia-smi

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sat Jun 13 01:19:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                

In [17]:
import birdnet

model = birdnet.load("acoustic", "2.4", "tf")

print("BirdNET model loaded")

BirdNET model loaded


In [18]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

# Current embedding pooling method.
# Keep this as "mean" for the first baseline.
# If you change this later, also change EMBEDDINGS_DIR to a new folder.
EMBEDDING_POOLING = "mean"

# BirdNET v2.4 embeddings are normally 1024-dimensional.
# The error 7168 = 7 * 1024 confirms some outputs are flattened window embeddings.
EXPECTED_BIRDNET_EMBEDDING_DIM = 1024


def convert_birdnet_result_to_numpy(result):
    """
    Converts BirdNET encode output into a 2D numpy array.

    Expected final shape:
        number_of_windows x embedding_dimension

    Fix included:
    - If BirdNET returns a flattened vector like 7168,
      reshape it into 7 x 1024 before pooling.
    """

    if isinstance(result, np.ndarray):
        arr = result

    elif isinstance(result, pd.DataFrame):
        numeric_df = result.select_dtypes(include=[np.number])
        arr = numeric_df.to_numpy()

    elif hasattr(result, "embeddings"):
        arr = np.asarray(result.embeddings)

    elif hasattr(result, "embedding"):
        arr = np.asarray(result.embedding)

    elif hasattr(result, "to_pandas"):
        pdf = result.to_pandas()
        numeric_df = pdf.select_dtypes(include=[np.number])
        arr = numeric_df.to_numpy()

    elif hasattr(result, "to_numpy"):
        arr = result.to_numpy()

    elif isinstance(result, dict):
        possible_keys = [
            "embeddings",
            "embedding",
            "features",
            "feature",
            "data",
            "values"
        ]

        arr = None

        for key in possible_keys:
            if key in result:
                arr = np.asarray(result[key])
                break

        if arr is None:
            raise ValueError(
                f"Could not find embeddings in BirdNET result dictionary. "
                f"Available keys: {list(result.keys())}"
            )

    else:
        raise TypeError(f"Unknown BirdNET encode output type: {type(result)}")

    arr = np.asarray(arr)

    if arr.size == 0:
        raise ValueError("BirdNET returned an empty embedding array.")

    arr = arr.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    # ----------------------------------------------------
    # Shape correction
    # ----------------------------------------------------

    if arr.ndim == 1:
        # Example: 7168 should become 7 x 1024
        if arr.size % EXPECTED_BIRDNET_EMBEDDING_DIM == 0:
            arr = arr.reshape(-1, EXPECTED_BIRDNET_EMBEDDING_DIM)
        else:
            raise ValueError(
                f"1D BirdNET embedding length {arr.size} is not divisible by "
                f"{EXPECTED_BIRDNET_EMBEDDING_DIM}. Cannot reshape safely."
            )

    elif arr.ndim == 2:
        # If shape is already windows x 1024, keep it.
        if arr.shape[1] == EXPECTED_BIRDNET_EMBEDDING_DIM:
            pass

        # If shape is 1 x flattened_windows, reshape it.
        elif arr.shape[0] == 1 and arr.shape[1] % EXPECTED_BIRDNET_EMBEDDING_DIM == 0:
            arr = arr.reshape(-1, EXPECTED_BIRDNET_EMBEDDING_DIM)

        # If shape is flattened_windows x 1, reshape it.
        elif arr.shape[1] == 1 and arr.shape[0] % EXPECTED_BIRDNET_EMBEDDING_DIM == 0:
            arr = arr.reshape(-1, EXPECTED_BIRDNET_EMBEDDING_DIM)

        else:
            raise ValueError(
                f"Unexpected 2D BirdNET embedding shape: {arr.shape}. "
                f"Expected second dimension {EXPECTED_BIRDNET_EMBEDDING_DIM}."
            )

    else:
        # Flatten everything except first axis, then check again.
        arr = arr.reshape(arr.shape[0], -1)

        if arr.shape[1] == EXPECTED_BIRDNET_EMBEDDING_DIM:
            pass
        elif arr.shape[1] % EXPECTED_BIRDNET_EMBEDDING_DIM == 0:
            arr = arr.reshape(-1, EXPECTED_BIRDNET_EMBEDDING_DIM)
        else:
            raise ValueError(
                f"Unexpected BirdNET embedding shape after reshape: {arr.shape}"
            )

    return arr


def pool_birdnet_windows(arr, pooling_method=EMBEDDING_POOLING):
    """
    Converts window-level BirdNET embeddings into one clip-level embedding.

    Supported options:
        mean
        max
        mean_max
        mean_std
        mean_max_std
    """

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape: {arr.shape}")

    if arr.shape[1] != EXPECTED_BIRDNET_EMBEDDING_DIM:
        raise ValueError(
            f"Expected embedding dimension {EXPECTED_BIRDNET_EMBEDDING_DIM}, "
            f"but got shape {arr.shape}"
        )

    if pooling_method == "mean":
        clip_embedding = arr.mean(axis=0)

    elif pooling_method == "max":
        clip_embedding = arr.max(axis=0)

    elif pooling_method == "mean_max":
        clip_embedding = np.concatenate([
            arr.mean(axis=0),
            arr.max(axis=0)
        ])

    elif pooling_method == "mean_std":
        clip_embedding = np.concatenate([
            arr.mean(axis=0),
            arr.std(axis=0)
        ])

    elif pooling_method == "mean_max_std":
        clip_embedding = np.concatenate([
            arr.mean(axis=0),
            arr.max(axis=0),
            arr.std(axis=0)
        ])

    else:
        raise ValueError(
            f"Unknown pooling method: {pooling_method}. "
            "Use one of: mean, max, mean_max, mean_std, mean_max_std"
        )

    clip_embedding = np.asarray(clip_embedding, dtype=np.float32)
    clip_embedding = np.nan_to_num(
        clip_embedding,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return clip_embedding


def extract_one_embedding(audio_path):
    """
    Extracts one clip-level BirdNET embedding for one audio file.

    audio_path should be local_filepath, not Google Drive filepath.
    """

    audio_path = Path(str(audio_path))

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    result = model.encode(str(audio_path))

    window_embeddings = convert_birdnet_result_to_numpy(result)

    clip_embedding = pool_birdnet_windows(
        window_embeddings,
        pooling_method=EMBEDDING_POOLING
    )

    return clip_embedding


def test_one_embedding_extraction(dataframe):
    """
    Quick test before running full extraction.
    Uses local_filepath if available, otherwise filepath.
    """

    if len(dataframe) == 0:
        raise ValueError("Dataframe is empty.")

    row = dataframe.iloc[0]

    if "local_filepath" in dataframe.columns:
        test_audio_path = row["local_filepath"]
    else:
        test_audio_path = row["filepath"]

    print("Testing BirdNET embedding extraction")
    print("Audio file:", test_audio_path)
    print("Pooling method:", EMBEDDING_POOLING)

    start_time = time.time()

    embedding = extract_one_embedding(test_audio_path)

    elapsed = time.time() - start_time

    print("Embedding shape:", embedding.shape)
    print("Embedding dtype:", embedding.dtype)
    print("First 10 values:", embedding[:10])
    print("Time taken:", round(elapsed, 3), "seconds")

    return embedding

In [19]:
test_embedding = test_one_embedding_extraction(train_df)

print("Embedding shape:", test_embedding.shape)
print("First 10 values:", test_embedding[:10])

Testing BirdNET embedding extraction
Audio file: /content/audio_assignment_runtime/train_audio/1161364/iNat1216197.ogg
Pooling method: mean
Embedding shape: (1024,)
Embedding dtype: float32
First 10 values: [0.17541146 0.2638661  0.4460199  0.32665685 0.37108445 0.33986208
 0.07368456 0.25656784 0.1486448  0.27002135]
Time taken: 3.116 seconds
Embedding shape: (1024,)
First 10 values: [0.17541146 0.2638661  0.4460199  0.32665685 0.37108445 0.33986208
 0.07368456 0.25656784 0.1486448  0.27002135]


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
from datetime import datetime
import time

# ----------------------------------------------------
# Embedding output setup
# ----------------------------------------------------
FORCE_REEXTRACT_CHUNKS = True
FORCE_REBUILD_FINAL_ARRAYS = True
# This should already be defined in Change 6.
# If not, default to mean pooling.
if "EMBEDDING_POOLING" not in globals():
    EMBEDDING_POOLING = "mean"

# Save embeddings in a folder named by pooling method.
# Example: embeddings/birdnet_original_mean
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings" / f"birdnet_original_{EMBEDDING_POOLING}"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Chunk folder: only a small number of chunk files are saved,
# instead of one cache file per audio file.
CHUNKS_DIR = EMBEDDINGS_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Number of audio files per chunk.
# 500 is a safe start. You can try 1000 later if stable.
CHUNK_SIZE = 500

# Set these to True only if you intentionally want to regenerate.
FORCE_REEXTRACT_CHUNKS = False
FORCE_REBUILD_FINAL_ARRAYS = False

print("Embeddings will be saved to:")
print(EMBEDDINGS_DIR)

print("\nChunk files will be saved to:")
print(CHUNKS_DIR)

print("\nPooling method:", EMBEDDING_POOLING)
print("Chunk size:", CHUNK_SIZE)


# ----------------------------------------------------
# Utility checks
# ----------------------------------------------------

def check_required_columns(dataframe, split_name):
    required_columns = ["label", "local_filepath", "relative_filepath"]

    missing_columns = [
        col for col in required_columns
        if col not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{split_name} dataframe is missing required columns: {missing_columns}\n"
            "Run the updated Change 4 cell first, because it creates local_filepath and relative_filepath."
        )

    missing_local_files = dataframe[
        ~dataframe["local_filepath"].apply(lambda p: Path(str(p)).exists())
    ]

    if len(missing_local_files) > 0:
        print(f"Missing local files in {split_name}:", len(missing_local_files))
        display(missing_local_files.head(10))

        raise FileNotFoundError(
            f"Some local files are missing for {split_name}. "
            "Check AUDIO_DIR and train_audio.zip extraction."
        )


check_required_columns(train_df, "train")
check_required_columns(val_df, "validation")


# ----------------------------------------------------
# Extract embeddings chunk by chunk
# ----------------------------------------------------

def extract_embeddings_to_chunks(dataframe, split_name, chunk_size=500):
    """
    Extracts BirdNET embeddings in chunks.

    For each chunk, saves:
    - one .npz file containing X and y
    - one metadata CSV
    - one failed CSV

    This is much faster on Google Drive than saving one file per audio sample.
    """

    split_chunk_dir = CHUNKS_DIR / split_name
    split_chunk_dir.mkdir(parents=True, exist_ok=True)

    total_rows = len(dataframe)
    number_of_chunks = (total_rows + chunk_size - 1) // chunk_size

    print("\n" + "=" * 80)
    print(f"Extracting {split_name} embeddings")
    print(f"Total rows: {total_rows}")
    print(f"Total chunks: {number_of_chunks}")
    print("=" * 80)

    failed_all_rows = []

    for chunk_id, start in enumerate(range(0, total_rows, chunk_size)):
        end = min(start + chunk_size, total_rows)

        chunk_npz_path = split_chunk_dir / f"{split_name}_chunk_{chunk_id:05d}_{start}_{end}.npz"
        chunk_meta_path = split_chunk_dir / f"{split_name}_chunk_{chunk_id:05d}_{start}_{end}_meta.csv"
        chunk_failed_path = split_chunk_dir / f"{split_name}_chunk_{chunk_id:05d}_{start}_{end}_failed.csv"

        if (
            chunk_npz_path.exists()
            and chunk_meta_path.exists()
            and not FORCE_REEXTRACT_CHUNKS
        ):
            print(f"Skipping existing chunk {chunk_id + 1}/{number_of_chunks}: rows {start}-{end}")
            continue

        chunk_df = dataframe.iloc[start:end].copy()

        X_chunk = []
        y_chunk = []
        local_paths = []
        relative_paths = []
        failed_rows = []

        print("\n" + "-" * 80)
        print(f"Processing {split_name} chunk {chunk_id + 1}/{number_of_chunks}: rows {start}-{end}")
        print("-" * 80)

        chunk_start_time = time.time()

        for row in tqdm(
            chunk_df.itertuples(index=False),
            total=len(chunk_df),
            desc=f"{split_name} chunk {chunk_id + 1}"
        ):
            local_path = str(row.local_filepath)
            relative_path = str(row.relative_filepath)
            label = str(row.label)

            try:
                emb = extract_one_embedding(local_path)

                emb = np.asarray(emb, dtype=np.float32)

                if emb.ndim != 1:
                    emb = emb.reshape(-1)

                X_chunk.append(emb)
                y_chunk.append(label)
                local_paths.append(local_path)
                relative_paths.append(relative_path)

            except Exception as e:
                print("Failed:", local_path)
                print("Error:", e)

                failed_rows.append({
                    "local_filepath": local_path,
                    "relative_filepath": relative_path,
                    "label": label,
                    "error": str(e)
                })

        if len(X_chunk) == 0:
            print("No successful embeddings in this chunk.")
            failed_df = pd.DataFrame(failed_rows)
            failed_df.to_csv(chunk_failed_path, index=False)
            failed_all_rows.extend(failed_rows)
            continue

        X_chunk = np.vstack(X_chunk).astype(np.float32)
        y_chunk = np.array(y_chunk)

        meta_chunk = pd.DataFrame({
            "local_filepath": local_paths,
            "relative_filepath": relative_paths,
            "label": y_chunk
        })

        # Save one chunk file.
        # Not compressed, because compression costs CPU time.
        np.savez(
            chunk_npz_path,
            X=X_chunk,
            y=y_chunk,
            local_filepath=np.array(local_paths),
            relative_filepath=np.array(relative_paths)
        )

        meta_chunk.to_csv(chunk_meta_path, index=False)

        failed_df = pd.DataFrame(failed_rows)
        failed_df.to_csv(chunk_failed_path, index=False)

        failed_all_rows.extend(failed_rows)

        elapsed = time.time() - chunk_start_time

        print("Saved chunk:")
        print(chunk_npz_path)
        print("Chunk X shape:", X_chunk.shape)
        print("Chunk y shape:", y_chunk.shape)
        print("Failed in this chunk:", len(failed_rows))
        print("Chunk time:", round(elapsed, 2), "seconds")

    failed_all_df = pd.DataFrame(failed_all_rows)
    failed_all_path = EMBEDDINGS_DIR / f"failed_{split_name}_all.csv"
    failed_all_df.to_csv(failed_all_path, index=False)

    print("\nFinished extracting chunks for:", split_name)
    print("Total failed files:", len(failed_all_df))
    print("Failed file saved to:")
    print(failed_all_path)


# ----------------------------------------------------
# Combine chunk files into final X/y/meta files
# ----------------------------------------------------

def combine_chunks(split_name):
    """
    Combines all saved chunk .npz files into:
    - X_<split>.npy
    - y_<split>.npy
    - meta_<split>.csv
    """

    split_chunk_dir = CHUNKS_DIR / split_name

    chunk_files = sorted(split_chunk_dir.glob(f"{split_name}_chunk_*.npz"))

    if len(chunk_files) == 0:
        raise FileNotFoundError(
            f"No chunk files found for {split_name} in {split_chunk_dir}"
        )

    print("\n" + "=" * 80)
    print(f"Combining {len(chunk_files)} chunk files for {split_name}")
    print("=" * 80)

    X_parts = []
    y_parts = []
    meta_parts = []

    for chunk_file in tqdm(chunk_files, desc=f"Combining {split_name} chunks"):
        data = np.load(chunk_file, allow_pickle=True)

        X_parts.append(data["X"])
        y_parts.append(data["y"])

        meta_file = Path(str(chunk_file).replace(".npz", "_meta.csv"))

        if meta_file.exists():
            meta_chunk = pd.read_csv(meta_file)
        else:
            meta_chunk = pd.DataFrame({
                "local_filepath": data["local_filepath"],
                "relative_filepath": data["relative_filepath"],
                "label": data["y"]
            })

        meta_parts.append(meta_chunk)

    X = np.vstack(X_parts).astype(np.float32)
    y = np.concatenate(y_parts)
    meta = pd.concat(meta_parts, ignore_index=True)

    X_path = EMBEDDINGS_DIR / f"X_{split_name}.npy"
    y_path = EMBEDDINGS_DIR / f"y_{split_name}.npy"
    meta_path = EMBEDDINGS_DIR / f"meta_{split_name}.csv"

    np.save(X_path, X)
    np.save(y_path, y)
    meta.to_csv(meta_path, index=False)

    print("\nSaved final files:")
    print(X_path)
    print(y_path)
    print(meta_path)

    print("\nFinal shapes:")
    print(f"X_{split_name}:", X.shape)
    print(f"y_{split_name}:", y.shape)
    print(f"meta_{split_name}:", meta.shape)

    return X, y, meta


# ----------------------------------------------------
# Load final files if already complete, otherwise extract/combine
# ----------------------------------------------------

def load_or_create_embeddings(dataframe, split_name):
    """
    If final X/y/meta files already exist, load them.
    Otherwise extract chunks and combine them.
    """

    X_path = EMBEDDINGS_DIR / f"X_{split_name}.npy"
    y_path = EMBEDDINGS_DIR / f"y_{split_name}.npy"
    meta_path = EMBEDDINGS_DIR / f"meta_{split_name}.csv"

    final_files_exist = (
        X_path.exists()
        and y_path.exists()
        and meta_path.exists()
    )

    if final_files_exist and not FORCE_REBUILD_FINAL_ARRAYS:
        print("\n" + "=" * 80)
        print(f"Final {split_name} embeddings already exist. Loading them.")
        print("=" * 80)

        X = np.load(X_path)
        y = np.load(y_path, allow_pickle=True)
        meta = pd.read_csv(meta_path)

        print(f"Loaded X_{split_name}:", X.shape)
        print(f"Loaded y_{split_name}:", y.shape)
        print(f"Loaded meta_{split_name}:", meta.shape)

        return X, y, meta

    extract_embeddings_to_chunks(
        dataframe=dataframe,
        split_name=split_name,
        chunk_size=CHUNK_SIZE
    )

    X, y, meta = combine_chunks(split_name)

    return X, y, meta


# ----------------------------------------------------
# Run train and validation extraction
# ----------------------------------------------------

X_train, y_train, meta_train = load_or_create_embeddings(train_df, "train")

X_validation, y_validation, meta_validation = load_or_create_embeddings(val_df, "validation")


# ----------------------------------------------------
# Save summary
# ----------------------------------------------------

failed_train_path = EMBEDDINGS_DIR / "failed_train_all.csv"
failed_validation_path = EMBEDDINGS_DIR / "failed_validation_all.csv"

failed_train_count = 0
failed_validation_count = 0

if failed_train_path.exists():
    failed_train_count = len(pd.read_csv(failed_train_path))

if failed_validation_path.exists():
    failed_validation_count = len(pd.read_csv(failed_validation_path))

summary = {
    "created_at": datetime.now().isoformat(),
    "embedding_type": "BirdNET acoustic embeddings",
    "birdnet_backend": "tf",
    "pooling": EMBEDDING_POOLING,
    "chunk_size": CHUNK_SIZE,
    "train_samples_saved": int(len(y_train)),
    "validation_samples_saved": int(len(y_validation)),
    "train_embedding_shape": list(X_train.shape),
    "validation_embedding_shape": list(X_validation.shape),
    "train_failed_files": int(failed_train_count),
    "validation_failed_files": int(failed_validation_count),
    "output_folder": str(EMBEDDINGS_DIR)
}

summary_path = EMBEDDINGS_DIR / "embedding_extraction_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSummary saved to:")
print(summary_path)

print(json.dumps(summary, indent=4))


# ----------------------------------------------------
# Final sanity check
# ----------------------------------------------------

print("\nFinal sanity check:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("meta_train:", meta_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("meta_validation:", meta_validation.shape)

print("\nEmbeddings are ready for Notebook 2.")

Embeddings will be saved to:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean

Chunk files will be saved to:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean/chunks

Pooling method: mean
Chunk size: 500

Extracting train embeddings
Total rows: 28357
Total chunks: 57
Skipping existing chunk 1/57: rows 0-500
Skipping existing chunk 2/57: rows 500-1000
Skipping existing chunk 3/57: rows 1000-1500
Skipping existing chunk 4/57: rows 1500-2000
Skipping existing chunk 5/57: rows 2000-2500
Skipping existing chunk 6/57: rows 2500-3000
Skipping existing chunk 7/57: rows 3000-3500
Skipping existing chunk 8/57: rows 3500-4000
Skipping existing chunk 9/57: rows 4000-4500
Skipping existing chunk 10/57: rows 4500-5000
Skipping existing chunk 11/57: rows 5000-5500
Skipping existing chunk 12/57: rows 5500-6000
Skipping existing chunk 13/57: rows 6000-6500
Skipping existing chunk 14/57: rows 6500-7000
Skipping existing chunk 15/57: rows 7000-7500
S

train chunk 24: 100%|██████████| 500/500 [28:47<00:00,  3.45s/it]


Saved chunk:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean/chunks/train/train_chunk_00023_11500_12000.npz
Chunk X shape: (500, 1024)
Chunk y shape: (500,)
Failed in this chunk: 0
Chunk time: 1727.17 seconds

--------------------------------------------------------------------------------
Processing train chunk 25/57: rows 12000-12500
--------------------------------------------------------------------------------


train chunk 25: 100%|██████████| 500/500 [33:16<00:00,  3.99s/it]


Saved chunk:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean/chunks/train/train_chunk_00024_12000_12500.npz
Chunk X shape: (500, 1024)
Chunk y shape: (500,)
Failed in this chunk: 0
Chunk time: 1996.2 seconds

--------------------------------------------------------------------------------
Processing train chunk 26/57: rows 12500-13000
--------------------------------------------------------------------------------


...

train chunk 27: 100%|██████████| 500/500 [31:23<00:00,  3.77s/it]


Saved chunk:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean/chunks/train/train_chunk_00026_13000_13500.npz
Chunk X shape: (500, 1024)
Chunk y shape: (500,)
Failed in this chunk: 0
Chunk time: 1883.36 seconds

--------------------------------------------------------------------------------
Processing train chunk 28/57: rows 13500-14000
--------------------------------------------------------------------------------


train chunk 28: 100%|██████████| 500/500 [33:06<00:00,  3.97s/it]


Saved chunk:
/content/drive/MyDrive/audio_assignment/zip/embeddings/birdnet_original_mean/chunks/train/train_chunk_00027_13500_14000.npz
Chunk X shape: (500, 1024)
Chunk y shape: (500,)
Failed in this chunk: 0
Chunk time: 1986.64 seconds

--------------------------------------------------------------------------------
Processing train chunk 29/57: rows 14000-14500
--------------------------------------------------------------------------------


train chunk 29:  72%|███████▏  | 359/500 [25:03<09:41,  4.12s/it]